In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_lakehouse = "lh_silver_usaspending"
gold_lakehouse = "lh_gold_usaspending"

# This is the verified business key for FactAward. It was widened from an
# original 5-column version after we found 17 real, distinct transaction
# rows sharing the same narrower key (different program_activity_name and
# different transaction_obligated_amount). This 10-column version was
# confirmed collision-free: COUNT(*) = COUNT(DISTINCT these 10 columns)
# on both bronze_assistance and bronze_contracts.
#
# IMPORTANT: measure columns (amounts) are deliberately NOT part of this key.
# If they were, a legitimate correction to an amount would produce a
# "different" key, causing MERGE to insert a duplicate row instead of
# updating the existing one.
fact_business_keys = [
    "award_unique_key",
    "parent_award_id_piid",
    "federal_account_symbol",
    "program_activity_code",
    "program_activity_name",
    "object_class_code",
    "direct_or_reimbursable_funding_source",
    "disaster_emergency_fund_code",
    "submission_period",
    "program_activity_reporting_key",
    "award_category",   # distinguishes assistance vs. contracts rows post-union
]

spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_lakehouse}.dbo")

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
# One row per Gold table we build, tracking how far incremental processing
# has gotten. Same pattern as Bronze's _bronze_load_ts and Silver's
# _silver_load_ts watermarks — just one layer further downstream.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_lakehouse}.dbo.gold_watermark (
        table_name STRING,
        last_watermark TIMESTAMP
    ) USING DELTA
""")

for tbl in ["dim_agency", "dim_account", "dim_program_activity", "dim_recipient", "fact_award"]:
    spark.sql(f"""
        INSERT INTO {gold_lakehouse}.dbo.gold_watermark
        SELECT '{tbl}', TIMESTAMP('1900-01-01')
        WHERE NOT EXISTS (
            SELECT 1 FROM {gold_lakehouse}.dbo.gold_watermark WHERE table_name = '{tbl}'
        )
    """)


def get_watermark(table_name: str):
    """Return the last successfully-processed _silver_load_ts for this Gold table."""
    row = spark.sql(f"""
        SELECT last_watermark FROM {gold_lakehouse}.dbo.gold_watermark
        WHERE table_name = '{table_name}'
    """).collect()
    return row[0]["last_watermark"] if row else "1900-01-01 00:00:00"


def set_watermark(table_name: str, new_ts):
    """Advance the watermark after a successful load. If this line never runs
    (e.g. the load raised an exception first), the next run safely reprocesses
    the same batch — the correct, safe failure mode for incremental pipelines."""
    spark.sql(f"""
        UPDATE {gold_lakehouse}.dbo.gold_watermark
        SET last_watermark = TIMESTAMP('{new_ts}')
        WHERE table_name = '{table_name}'
    """)

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 4, Finished, Available, Finished, False)

In [3]:
def get_unioned_silver(watermark_col="_silver_load_ts", since=None):
    """
    Combines silver_assistance and silver_contracts into one DataFrame.
    They don't share every column (e.g. cfda_number is assistance-only,
    naics_code is contracts-only), so we align both DataFrames to the same
    full column set first, filling missing columns with NULL, before unioning.
    An AwardCategory column is added so downstream code (and the fact table)
    can always tell which source table a row originally came from.
    """
    a = spark.table(f"{silver_lakehouse}.dbo.silver_assistance").withColumn("award_category", F.lit("assistance"))
    c = spark.table(f"{silver_lakehouse}.dbo.silver_contracts").withColumn("award_category", F.lit("contracts"))

    all_cols = sorted(set(a.columns) | set(c.columns))
    for col in all_cols:
        if col not in a.columns:
            a = a.withColumn(col, F.lit(None))
        if col not in c.columns:
            c = c.withColumn(col, F.lit(None))

    unioned = a.select(*all_cols).unionByName(c.select(*all_cols))

    # Incremental filter: only rows Silver has touched since Gold's last run.
    if since is not None:
        unioned = unioned.filter(F.col(watermark_col) > since)
    return unioned

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 5, Finished, Available, Finished, False)

In [4]:
def ensure_unknown_member(dim_name: str, business_key, attribute_cols: list):
    """
    Guarantees an 'Unknown' member with surrogate key -1 exists in this
    dimension, so fact rows with a missing/unmatched business key (e.g. the
    358,626 rows with NULL program_activity_code, confirmed as genuine source
    gaps, not a join bug) can point at a real dimension row instead of NULL.
    Idempotent: safe to call on every run, whether the table is brand new
    or already populated.

    business_key can be a single column name (str) or a list of column
    names (composite key, e.g. dim_program_activity's [code, name] pair).
    """
    table_name = f"{gold_lakehouse}.dbo.{dim_name}"
    surrogate_key = f"{dim_name.replace('dim_', '')}_key"
    key_cols = business_key if isinstance(business_key, list) else [business_key]

    exists = spark.sql(f"""
        SELECT COUNT(*) AS c FROM {table_name} WHERE {surrogate_key} = -1
    """).collect()[0]["c"]

    if exists == 0:
        print(f"{dim_name}: inserting Unknown member (key = -1)")
        # -1 for the surrogate key, then "UNKNOWN" for every key column
        # (one per column in key_cols), then "Unknown" for every attribute column.
        unknown_row_values = [-1] + ["UNKNOWN"] * len(key_cols) + ["Unknown"] * len(attribute_cols)
        unknown_cols = [surrogate_key] + key_cols + attribute_cols
        unknown_df = spark.createDataFrame([tuple(unknown_row_values)], unknown_cols) \
            .withColumn(surrogate_key, F.col(surrogate_key).cast("long"))   # <-- force type match
        unknown_df.write.format("delta").mode("append").saveAsTable(table_name)


def merge_dimension(dim_name: str, business_key, attribute_cols: list, source_df):
    """
    Builds/updates one dimension table, assigning a surrogate key to any
    newly-seen business-key value.

    business_key can be a single column name (str) or a list of column
    names (composite key). dim_program_activity needs a composite key
    [program_activity_code, program_activity_name], since the same code
    can legitimately pair with different names (confirmed via collision
    checks) — program_activity_code alone is not unique.

    Surrogate key note: we do NOT use monotonically_increasing_id() directly
    as the final key. That function's output is not stable across separate
    Spark jobs/runs (it depends on partition layout at execution time), so
    using it directly on every incremental run risks a new row getting a key
    value that collides with one assigned in an earlier run. Instead, new
    keys are computed as (current max existing key) + a deterministic
    row_number() over the new rows only — guaranteed unique and always
    increasing, no matter how many times this runs.
    """
    table_name = f"{gold_lakehouse}.dbo.{dim_name}"
    surrogate_key = f"{dim_name.replace('dim_', '')}_key"
    key_cols = business_key if isinstance(business_key, list) else [business_key]

    # One row per distinct business-key combination seen in this batch.
    new_members = (source_df
        .select(*key_cols, *attribute_cols)
        .filter(F.col(key_cols[0]).isNotNull())
        .dropDuplicates(key_cols))

    if not spark.catalog.tableExists(table_name):
        # First-ever load of this dimension: every member is "new".
        print(f"Creating {table_name} (first load)")
        w = Window.orderBy(*key_cols)
        result = (new_members
            .withColumn(surrogate_key, F.row_number().over(w).cast("long"))) 
        result.write.format("delta").mode("overwrite").saveAsTable(table_name)
        ensure_unknown_member(dim_name, business_key, attribute_cols)
        return

    ensure_unknown_member(dim_name, business_key, attribute_cols)

    existing = spark.table(table_name)
    existing_keys = existing.select(*key_cols).distinct()

    # Only business-key combinations not already present get a new surrogate key.
    truly_new = new_members.join(existing_keys, key_cols, "left_anti")

    if truly_new.count() == 0:
        print(f"{dim_name}: no new members")
        return

    max_key = existing.agg(F.max(surrogate_key)).collect()[0][0] or 0

    w = Window.orderBy(*key_cols)
    numbered = (truly_new
        .withColumn("rn", F.row_number().over(w))
        .withColumn(surrogate_key, (F.col("rn") + max_key).cast("long"))
        .drop("rn"))

    print(f"{dim_name}: inserting {numbered.count()} new member(s)")
    (numbered
        .select(surrogate_key, *key_cols, *attribute_cols)
        .write.format("delta").mode("append").saveAsTable(table_name))

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 6, Finished, Available, Finished, False)

In [5]:
# Dimensions share one practical watermark (dim_agency's) since they're all
# sourced from the same unioned Silver read in this design.
watermark = get_watermark("dim_agency")
src = get_unioned_silver(since=watermark)

merge_dimension("dim_agency", "agency_code",
                ["awarding_agency_name", "funding_agency_name"], src)

merge_dimension("dim_account", "federal_account_symbol",
                ["federal_account_name", "budget_function", "budget_subfunction",
                 "object_class_code", "object_class_name"], src)

merge_dimension("dim_program_activity", ["program_activity_code", "program_activity_name"], [], src)

merge_dimension("dim_recipient", "recipient_uei",
                ["recipient_name", "recipient_state", "recipient_country", "recipient_city"], src)

# Advance all four dimension watermarks together, since they were built from
# the same incremental Silver read.
max_ts = src.agg(F.max("_silver_load_ts")).collect()[0][0]
if max_ts:
    for d in ["dim_agency", "dim_account", "dim_program_activity", "dim_recipient"]:
        set_watermark(d, max_ts)

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 7, Finished, Available, Finished, False)

Creating lh_gold_usaspending.dbo.dim_agency (first load)
dim_agency: inserting Unknown member (key = -1)
Creating lh_gold_usaspending.dbo.dim_account (first load)
dim_account: inserting Unknown member (key = -1)
Creating lh_gold_usaspending.dbo.dim_program_activity (first load)
dim_program_activity: inserting Unknown member (key = -1)
Creating lh_gold_usaspending.dbo.dim_recipient (first load)
dim_recipient: inserting Unknown member (key = -1)


In [6]:
fact_watermark = get_watermark("fact_award")
fact_src = get_unioned_silver(since=fact_watermark)

if fact_src.count() > 0:
    # Look up each dimension's surrogate key so the fact table stores compact
    # integer foreign keys instead of repeating full text attributes per row.
    dim_agency = spark.table(f"{gold_lakehouse}.dbo.dim_agency").select("agency_key", "agency_code")
    dim_account = spark.table(f"{gold_lakehouse}.dbo.dim_account").select("account_key", "federal_account_symbol")
    dim_pa = spark.table(f"{gold_lakehouse}.dbo.dim_program_activity").select(
    "program_activity_key", "program_activity_code", "program_activity_name")
    dim_recipient = spark.table(f"{gold_lakehouse}.dbo.dim_recipient").select("recipient_key", "recipient_uei")

    fact_df = (fact_src
        .join(dim_agency, "agency_code", "left")
        .join(dim_account, "federal_account_symbol", "left")
        .join(dim_pa, ["program_activity_code", "program_activity_name"], "left")
        .join(dim_recipient, "recipient_uei", "left")
        .select(
            "agency_key", "account_key", "program_activity_key", "recipient_key",
            *fact_business_keys,
            "transaction_obligated_amount",
            "gross_outlay_amount_FYB_to_period_end",
            "last_modified_date",
            "agency_code",   # kept as a plain column too — this is what the RLS rule filters on
        )
        .withColumn("program_activity_key", F.coalesce(F.col("program_activity_key"), F.lit(-1)))
        .withColumn("recipient_key", F.coalesce(F.col("recipient_key"), F.lit(-1)))
        .withColumn("fiscal_year", F.concat(F.lit("FY"), F.substring(F.col("submission_period"), 3, 4)))
        .withColumn("period_number", F.substring(F.col("submission_period"), 8, 2).cast("int"))
        )

    fact_table = f"{gold_lakehouse}.dbo.fact_award"

    if not spark.catalog.tableExists(fact_table):
        print("Creating fact_award (first load)")
        fact_df.write.format("delta").mode("overwrite").saveAsTable(fact_table)
    else:
        print("Merging into fact_award")
        gold_fact = DeltaTable.forName(spark, fact_table)

        # <=> is null-safe equality: several of these key columns (e.g.
        # program_activity_reporting_key) can legitimately be NULL, and plain
        # = never matches two NULLs against each other.
        join_cond = " AND ".join(f"t.{k} <=> s.{k}" for k in fact_business_keys)

        (gold_fact.alias("t")
            .merge(fact_df.alias("s"), join_cond)
            # Only overwrite if the incoming row is genuinely newer — this is
            # what prevents an out-of-order/late-arriving batch from
            # clobbering more recent data with stale values.
            .whenMatchedUpdateAll(condition="s.last_modified_date > t.last_modified_date")
            .whenNotMatchedInsertAll()
            .execute())

    max_ts = fact_src.agg(F.max("_silver_load_ts")).collect()[0][0]
    set_watermark("fact_award", max_ts)
else:
    print("No new fact rows to process")

StatementMeta(, 30f2e1dc-eb5c-42ed-85be-7a3f2ebc3409, 8, Finished, Available, Finished, False)

Creating fact_award (first load)
